# GroundedX revision runs (Kaggle, 2x T4)

**Settings: Accelerator = GPU T4 x2, Internet = ON.** Edit `PHASES` below, then *Save Version -> Save & Run All*.

This notebook is only a launcher: it clones the `revision-minds` branch and runs
`scripts/kaggle_session.py`, which holds all the logic. **You never need to re-import this notebook.**

- `PHASES = ["pilot"]`  -> hybrid validation pilot (~1 h)
- `PHASES = ["hybrid"]` -> hybrid test runs + their profiling (~4-5 h)

Finished jobs are skipped automatically. Within the first minutes the log prints a **PLAN** listing
the pending jobs per phase - check it. Optional: attach a previous version's output as input to reuse
the compiled llama-server and models (saves ~20 min) and to resume unfinished jobs.
At the end, download `results_v2.zip` from the Output tab.

In [ ]:
PHASES = ["pilot"]
DEADLINE_HOURS = 11.0

In [ ]:
import os, shutil, subprocess, sys
REPO = "/kaggle/working/groundedx"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", "revision-minds",
                       "https://github.com/TanishqX-404/groundedx.git", REPO])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO,
                       "nvidia-ml-py", "xgboost", "huggingface_hub", "pytest"])
print(subprocess.check_output(["git", "-C", REPO, "log", "--oneline", "-1"], text=True))
proc = subprocess.Popen([sys.executable, "-u", f"{REPO}/scripts/kaggle_session.py",
                         "--phases", *PHASES, "--deadline-hours", str(DEADLINE_HOURS)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=REPO)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit code", proc.wait())